## Imports and loading

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import MinMaxScaler

from sklearn.impute import SimpleImputer
import html
import re

print("Caricamento dati...")

train_df = pd.read_csv("development.csv")
train_df['title'] = train_df['title'].fillna('')
train_df['article'] = train_df['article'].fillna('')
train_df['text_content'] = train_df['title']+ ' ' + train_df['article']


eval_df = pd.read_csv("evaluation.csv")

eval_df['title'] = eval_df['title'].fillna('')
eval_df['article'] = eval_df['article'].fillna('')
eval_df['text_content'] = eval_df['title']+ ' '  + eval_df['article']

print("Dati caricati.")

import re
import html

def clean_text(text):
    if not isinstance(text, str):
        return "" # Gestione robusta dei NaN

    # 1. Decodifica entità HTML (es. &quot; -> ")
    text = html.unescape(text)
    
    # 2. Rimozione Tag HTML (es. <br>, <p>, <div>)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    
    # 4. Rimozione Metadata editoriali tra parentesi, rimuove "(Reuters)", "(AP)", "(CNN)".
    # Evita che il modello impari la fonte invece del contenuto.
    text = re.sub(r'\([A-Za-z\s\.]+\)', ' ', text)
    
    # 6. Rimozione caratteri singoli isolati
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
    
    # 7. Normalizzazione Spazi e Lowercase
    text = re.sub(r'\s+', ' ', text).strip()
    
    
    return text

train_df['text_content'] = train_df['text_content'].apply(clean_text)
eval_df['text_content'] = eval_df['text_content'].apply(clean_text)


Caricamento dati...
Dati caricati.


## Preprocessing

In [7]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder


text_transformer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=3,
    max_df=0.5
    )

cat_transformer = OneHotEncoder(handle_unknown='ignore')

hour_transformer = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ])
source_pipe = Pipeline([
    ('encoder', OneHotEncoder(max_categories=51, handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('txt', text_transformer, 'text_content'),
        ('ord', cat_transformer, ['page_rank']),
        ('hour', hour_transformer, ['hour_cos','hour_sin']),
        ('cat_source', source_pipe, ['source']),
        
    ]
)
train_df['day_of_week'] = pd.to_datetime(train_df['timestamp'], errors='coerce').dt.dayofweek
eval_df['day_of_week'] = pd.to_datetime(eval_df['timestamp'], errors='coerce').dt.dayofweek

train_df['is_weekend'] = train_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
train_df['timestamp_missing'] = train_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)
eval_df['is_weekend'] = eval_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
eval_df['timestamp_missing'] = eval_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)


train_df['hour_of_day'] = pd.to_datetime(train_df['timestamp'], errors='coerce').dt.hour
eval_df['hour_of_day'] = pd.to_datetime(eval_df['timestamp'], errors='coerce').dt.hour
train_df['hour_sin'] = np.sin(2 * np.pi * train_df['hour_of_day'] / 24)
train_df['hour_cos'] = np.cos(2 * np.pi * train_df['hour_of_day'] / 24)

eval_df['hour_sin'] = np.sin(2 * np.pi * eval_df['hour_of_day'] / 24)
eval_df['hour_cos'] = np.cos(2 * np.pi * eval_df['hour_of_day'] / 24)

X_eval = eval_df[['text_content', 'source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]


X = train_df[['text_content','source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]
y = train_df['label']


## Linear SVC

In [8]:

 ## LINEAR SVC MODEL

classifier = LinearSVC(random_state=42)
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', classifier)
])


param_grid = {
    'clf__C': [0.18],
    'clf__class_weight': ['balanced'],
    'clf__loss':['squared_hinge'],
    'preprocessor__txt__ngram_range': [(1,3)],
    'preprocessor__cat_source__encoder__max_categories': [301] # ora è fissato a 300, poi mettiamo più valori

}

grid_search = GridSearchCV(
    model_pipeline, 
    param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1, 
    verbose=2
)

grid_search.fit(X, y)
print("LINEAR SVC - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_



Fitting 5 folds for each of 1 candidates, totalling 5 fits
LINEAR SVC - Grid Search Results:
{'clf__C': 0.18, 'clf__class_weight': 'balanced', 'clf__loss': 'squared_hinge', 'preprocessor__cat_source__encoder__max_categories': 301, 'preprocessor__txt__ngram_range': (1, 3)}
Best Cross-Validation Macro F1: 0.7209


In [9]:
best_model.fit(X, y)
y_eval_pred = best_model.predict(X_eval)

pd.DataFrame({
    'Id': eval_df['Id'],
    'Predicted': y_eval_pred}).to_csv('submission.csv', index=False)

# HistGradientBoosting


In [10]:
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import KNNImputer
from sklearn.pipeline import FunctionTransformer
from sklearn.preprocessing import OrdinalEncoder

to_dense = FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)
text_transformer = Pipeline([
    ('tfidf', TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3),
    sublinear_tf=True,
    max_features=2000
    )),
    ('to_dense', to_dense)
])





cat_transformer = OneHotEncoder(handle_unknown='ignore')

hour_transformer = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ])
source_pipe = Pipeline([
    ('encoder', OneHotEncoder(max_categories=301, handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('txt', text_transformer, 'text_content'),
        ('ord', cat_transformer, ['page_rank']),
        ('hour', hour_transformer, ['hour_cos','hour_sin']),
        ('cat_source', source_pipe, ['source']),
    ]
)
train_df['day_of_week'] = pd.to_datetime(train_df['timestamp'], errors='coerce').dt.dayofweek
eval_df['day_of_week'] = pd.to_datetime(eval_df['timestamp'], errors='coerce').dt.dayofweek

train_df['is_weekend'] = train_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
train_df['timestamp_missing'] = train_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)
eval_df['is_weekend'] = eval_df['day_of_week'].apply(lambda x: 1 if x >=5 else 0)
eval_df['timestamp_missing'] = eval_df['day_of_week'].apply(lambda x: 1 if pd.isna(x) else 0)

train_df['text_lenght'] = train_df['text_content'].apply(lambda x: len(x))
eval_df['text_lenght'] = eval_df['text_content'].apply(lambda x: len(x))
train_df['hour_of_day'] = pd.to_datetime(train_df['timestamp'], errors='coerce').dt.hour
eval_df['hour_of_day'] = pd.to_datetime(eval_df['timestamp'], errors='coerce').dt.hour
train_df['hour_sin'] = np.sin(2 * np.pi * train_df['hour_of_day'] / 24)
train_df['hour_cos'] = np.cos(2 * np.pi * train_df['hour_of_day'] / 24)

eval_df['hour_sin'] = np.sin(2 * np.pi * eval_df['hour_of_day'] / 24)
eval_df['hour_cos'] = np.cos(2 * np.pi * eval_df['hour_of_day'] / 24)

X_eval = eval_df[['text_content', 'source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]


X = train_df[['text_content','source','page_rank','is_weekend','timestamp_missing','hour_sin','hour_cos']]
y = train_df['label']


In [ ]:


from sklearn.ensemble import HistGradientBoostingClassifier

classifier = HistGradientBoostingClassifier(random_state=42, early_stopping=True)
model_pipeline = Pipeline([
    ('clf', classifier)
])
X=preprocessor.fit_transform(X)
X_eval=preprocessor.transform(X_eval)


param_grid = {
    'learning_rate': [0.01, 0.1],       # Quanto velocemente impara
    'max_iter': [100, 200],             # Numero massimo di alberi (simile a n_estimators)
    'max_depth': [None, 3, 5],               # Profondità massima (None = illimitata)
    'min_samples_leaf': [20, 40],            # Minimo numero di campioni per foglia (regolarizzazione)
    'l2_regularization': [0, 0.1, 1.0],      # Regolarizzazione L2 (Ridge)
    #'preprocessor__txt__tfidf__ngram_range': [(1,3)],
    #'preprocessor__cat_source__imputer_top_n__top_n': [50, 300] # ora è fissato a 300, poi mettiamo più valori

}

grid_search = GridSearchCV(
    classifier, 
    param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1, 
    verbose=2
)

grid_search.fit(X, y)
print("LINEAR SVC - Grid Search Results:")
print(grid_search.best_params_)
print(f"Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_



Fitting 5 folds for each of 72 candidates, totalling 360 fits


KeyboardInterrupt: 